# Personalized Learning Path Generator - AI-Powered Education System using spaCy NLP

**Framework:** [spaCy](https://spacy.io/) - Industrial-strength Natural Language Processing library

**Description:** An intelligent learning path generator that creates personalized educational journeys based on individual learning styles, goals, current knowledge level, and preferences. This system uses advanced NLP techniques to analyze learning content, assess knowledge gaps, and recommend optimal learning sequences across multiple domains and formats.

**Key Features:**
- Learning style assessment and profiling
- Knowledge gap analysis using NLP
- Adaptive learning path generation
- Multi-modal content recommendation
- Progress tracking and path optimization
- Skill dependency mapping

**Use Cases:**
- Corporate training programs
- Academic curriculum planning
- Professional skill development
- Certification preparation
- Self-directed learning optimization

## Installation

```bash
pip install spacy
pip install scikit-learn
pip install pandas
pip install numpy
pip install matplotlib
pip install seaborn
pip install networkx
pip install requests
pip install python-dotenv
python -m spacy download en_core_web_sm
```

In [ ]:
import os
import json
import random
import logging
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import math

try:
    import spacy
    from spacy.matcher import Matcher
    from spacy.util import filter_spans
except ImportError:
    print("Warning: spacy not installed. Install with: pip install spacy")
    spacy = None

try:
    import pandas as pd
    import numpy as np
except ImportError:
    print("Warning: pandas/numpy not installed. Install with: pip install pandas numpy")
    pd = None
    np = None

try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
    from sklearn.cluster import KMeans
except ImportError:
    print("Warning: scikit-learn not installed. Install with: pip install scikit-learn")
    TfidfVectorizer = None
    cosine_similarity = None
    KMeans = None

try:
    import networkx as nx
except ImportError:
    print("Warning: networkx not installed. Install with: pip install networkx")
    nx = None

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize spaCy model
if spacy:
    try:
        nlp = spacy.load("en_core_web_sm")
    except OSError:
        print("Warning: spaCy model not found. Run: python -m spacy download en_core_web_sm")
        nlp = None
else:
    nlp = None

In [ ]:
class LearningStyle(Enum):
    VISUAL = "visual"
    AUDITORY = "auditory"
    KINESTHETIC = "kinesthetic"
    READING_WRITING = "reading_writing"

class ContentType(Enum):
    VIDEO = "video"
    ARTICLE = "article"
    INTERACTIVE = "interactive"
    QUIZ = "quiz"
    PROJECT = "project"
    PODCAST = "podcast"
    BOOK = "book"
    COURSE = "course"

class DifficultyLevel(Enum):
    BEGINNER = "beginner"
    INTERMEDIATE = "intermediate"
    ADVANCED = "advanced"
    EXPERT = "expert"

@dataclass
class LearningGoal:
    """Represents a specific learning objective"""
    goal_id: str
    title: str
    description: str
    target_skills: List[str]
    difficulty_level: DifficultyLevel
    estimated_hours: int
    priority: int = 1  # 1-10 scale
    deadline: Optional[datetime] = None

@dataclass
class LearningResource:
    """Represents a learning resource/content"""
    resource_id: str
    title: str
    description: str
    content_type: ContentType
    difficulty_level: DifficultyLevel
    skills_covered: List[str]
    prerequisites: List[str]
    duration_minutes: int
    url: Optional[str] = None
    rating: Optional[float] = None
    learning_styles: List[LearningStyle] = field(default_factory=list)
    tags: List[str] = field(default_factory=list)

@dataclass
class LearnerProfile:
    """Represents a learner's profile and preferences"""
    learner_id: str
    name: str
    learning_styles: List[LearningStyle]
    current_skills: Dict[str, float]  # skill -> proficiency (0-1)
    goals: List[LearningGoal]
    preferred_content_types: List[ContentType]
    available_hours_per_week: int
    preferred_session_length: int  # minutes
    learning_pace: str = "medium"  # slow, medium, fast
    interests: List[str] = field(default_factory=list)
    completed_resources: List[str] = field(default_factory=list)
    learning_history: List[Dict] = field(default_factory=list)

@dataclass
class LearningPathStep:
    """Represents a single step in a learning path"""
    step_id: str
    resource: LearningResource
    estimated_completion_time: datetime
    prerequisites_met: bool
    importance_score: float
    personalization_score: float
    notes: str = ""

@dataclass
class LearningPath:
    """Represents a complete learning path"""
    path_id: str
    learner_id: str
    goal: LearningGoal
    steps: List[LearningPathStep]
    estimated_total_hours: int
    estimated_completion_date: datetime
    confidence_score: float
    created_at: datetime = field(default_factory=datetime.now)
    last_updated: datetime = field(default_factory=datetime.now)

In [ ]:
class NLPAnalyzer:
    """NLP-powered content analysis and skill extraction"""
    
    def __init__(self):
        self.nlp = nlp
        self.skill_matcher = None
        self.tfidf_vectorizer = None
        self.skill_vectors = {}
        
        if self.nlp:
            self._setup_skill_matcher()
        
        if TfidfVectorizer:
            self.tfidf_vectorizer = TfidfVectorizer(
                max_features=1000,
                stop_words='english',
                ngram_range=(1, 3)
            )
    
    def _setup_skill_matcher(self):
        """Setup pattern matching for skill extraction"""
        if not self.nlp:
            return
            
        self.skill_matcher = Matcher(self.nlp.vocab)
        
        # Define patterns for common skills
        skill_patterns = [
            ["programming", "coding", "development"],
            ["data", "analysis", "analytics"],
            ["machine", "learning", "AI"],
            ["project", "management"],
            ["communication", "presentation"],
            ["leadership", "management"],
            ["design", "UX", "UI"],
            ["marketing", "digital", "social"],
            ["finance", "accounting"],
            ["statistics", "mathematics"]
        ]
        
        for i, pattern_words in enumerate(skill_patterns):
            for word in pattern_words:
                pattern = [{"LOWER": word.lower()}]
                self.skill_matcher.add(f"SKILL_{i}_{word}", [pattern])
    
    def extract_skills_from_text(self, text: str) -> List[str]:
        """Extract skills mentioned in text using NLP"""
        if not self.nlp or not text:
            return []
        
        doc = self.nlp(text)
        skills = set()
        
        # Extract using pattern matching
        if self.skill_matcher:
            matches = self.skill_matcher(doc)
            for match_id, start, end in matches:
                skill = doc[start:end].text.lower()
                skills.add(skill)
        
        # Extract noun phrases that might be skills
        for chunk in doc.noun_chunks:
            if len(chunk.text.split()) <= 3:  # Limit to reasonable skill names
                chunk_text = chunk.text.lower().strip()
                if self._is_likely_skill(chunk_text):
                    skills.add(chunk_text)
        
        # Extract named entities that might be technologies or skills
        for ent in doc.ents:
            if ent.label_ in ["ORG", "PRODUCT", "LANGUAGE"] and len(ent.text.split()) <= 2:
                skills.add(ent.text.lower())
        
        return list(skills)
    
    def _is_likely_skill(self, text: str) -> bool:
        """Determine if text is likely a skill"""
        skill_keywords = [
            "programming", "coding", "development", "analysis", "management",
            "design", "marketing", "communication", "leadership", "strategy",
            "optimization", "automation", "testing", "debugging", "research",
            "planning", "coordination", "documentation", "training", "support"
        ]
        
        return any(keyword in text for keyword in skill_keywords)
    
    def analyze_content_difficulty(self, content: str) -> DifficultyLevel:
        """Analyze content difficulty using NLP features"""
        if not self.nlp or not content:
            return DifficultyLevel.INTERMEDIATE
        
        doc = self.nlp(content)
        
        # Calculate complexity metrics
        avg_sentence_length = np.mean([len(sent.text.split()) for sent in doc.sents]) if doc.sents else 0
        avg_word_length = np.mean([len(token.text) for token in doc if token.is_alpha]) if doc else 0
        technical_terms = len([token for token in doc if token.is_alpha and len(token.text) > 8])
        
        # Calculate difficulty score
        difficulty_score = (
            (avg_sentence_length - 15) * 0.3 +
            (avg_word_length - 5) * 0.4 +
            (technical_terms / len(doc) * 100) * 0.3
        )
        
        if difficulty_score < -2:
            return DifficultyLevel.BEGINNER
        elif difficulty_score < 2:
            return DifficultyLevel.INTERMEDIATE
        elif difficulty_score < 5:
            return DifficultyLevel.ADVANCED
        else:
            return DifficultyLevel.EXPERT
    
    def calculate_content_similarity(self, content1: str, content2: str) -> float:
        """Calculate similarity between two pieces of content"""
        if not self.tfidf_vectorizer or not content1 or not content2:
            return 0.0
        
        try:
            vectors = self.tfidf_vectorizer.fit_transform([content1, content2])
            similarity = cosine_similarity(vectors[0:1], vectors[1:2])[0][0]
            return float(similarity)
        except Exception as e:
            logger.warning(f"Error calculating similarity: {e}")
            return 0.0
    
    def extract_learning_objectives(self, content: str) -> List[str]:
        """Extract learning objectives from content description"""
        if not self.nlp or not content:
            return []
        
        doc = self.nlp(content)
        objectives = []
        
        # Look for objective patterns
        objective_patterns = [
            "learn", "understand", "master", "develop", "build", "create",
            "implement", "analyze", "evaluate", "design", "solve", "apply"
        ]
        
        for sent in doc.sents:
            sent_text = sent.text.lower()
            if any(pattern in sent_text for pattern in objective_patterns):
                objectives.append(sent.text.strip())
        
        return objectives[:5]  # Limit to top 5 objectives
    
    def analyze_prerequisite_relationships(self, skills: List[str]) -> Dict[str, List[str]]:
        """Analyze prerequisite relationships between skills"""
        prerequisites = {}
        
        # Define some common prerequisite relationships
        prerequisite_rules = {
            "advanced programming": ["programming", "coding"],
            "machine learning": ["programming", "statistics", "mathematics"],
            "data analysis": ["statistics", "programming"],
            "project management": ["communication", "planning"],
            "leadership": ["communication", "management"],
            "system design": ["programming", "architecture"],
            "database design": ["programming", "data modeling"],
            "web development": ["programming", "html", "css"]
        }
        
        for skill in skills:
            skill_lower = skill.lower()
            prerequisites[skill] = []
            
            for advanced_skill, prereqs in prerequisite_rules.items():
                if advanced_skill in skill_lower:
                    prerequisites[skill].extend(prereqs)
        
        return prerequisites

In [ ]:
class LearningPathGenerator:
    """Core learning path generation engine"""
    
    def __init__(self):
        self.nlp_analyzer = NLPAnalyzer()
        self.resources_database = []
        self.skill_graph = None
        
        # Initialize with sample resources
        self._initialize_sample_resources()
        
        # Build skill dependency graph
        if nx:
            self._build_skill_graph()
    
    def _initialize_sample_resources(self):
        """Initialize with sample learning resources"""
        sample_resources = [
            LearningResource(
                resource_id="py_basics_1",
                title="Python Fundamentals",
                description="Learn Python programming basics including variables, data types, and control structures",
                content_type=ContentType.COURSE,
                difficulty_level=DifficultyLevel.BEGINNER,
                skills_covered=["python", "programming", "variables", "data types"],
                prerequisites=[],
                duration_minutes=300,
                learning_styles=[LearningStyle.VISUAL, LearningStyle.READING_WRITING],
                rating=4.5,
                tags=["programming", "beginner", "python"]
            ),
            LearningResource(
                resource_id="ds_intro_1",
                title="Introduction to Data Science",
                description="Overview of data science concepts, tools, and methodologies",
                content_type=ContentType.VIDEO,
                difficulty_level=DifficultyLevel.BEGINNER,
                skills_covered=["data science", "statistics", "data analysis"],
                prerequisites=[],
                duration_minutes=120,
                learning_styles=[LearningStyle.VISUAL, LearningStyle.AUDITORY],
                rating=4.3,
                tags=["data science", "beginner", "overview"]
            ),
            LearningResource(
                resource_id="ml_basics_1",
                title="Machine Learning Fundamentals",
                description="Introduction to machine learning algorithms and applications",
                content_type=ContentType.COURSE,
                difficulty_level=DifficultyLevel.INTERMEDIATE,
                skills_covered=["machine learning", "algorithms", "supervised learning"],
                prerequisites=["python", "statistics", "mathematics"],
                duration_minutes=480,
                learning_styles=[LearningStyle.VISUAL, LearningStyle.KINESTHETIC],
                rating=4.7,
                tags=["machine learning", "intermediate", "algorithms"]
            ),
            LearningResource(
                resource_id="pm_essentials_1",
                title="Project Management Essentials",
                description="Core project management principles and methodologies",
                content_type=ContentType.ARTICLE,
                difficulty_level=DifficultyLevel.BEGINNER,
                skills_covered=["project management", "planning", "coordination"],
                prerequisites=["communication"],
                duration_minutes=90,
                learning_styles=[LearningStyle.READING_WRITING],
                rating=4.2,
                tags=["project management", "beginner", "planning"]
            ),
            LearningResource(
                resource_id="web_dev_1",
                title="Web Development Bootcamp",
                description="Complete web development course covering HTML, CSS, and JavaScript",
                content_type=ContentType.INTERACTIVE,
                difficulty_level=DifficultyLevel.INTERMEDIATE,
                skills_covered=["web development", "html", "css", "javascript"],
                prerequisites=["programming"],
                duration_minutes=600,
                learning_styles=[LearningStyle.KINESTHETIC, LearningStyle.VISUAL],
                rating=4.6,
                tags=["web development", "intermediate", "full stack"]
            ),
            LearningResource(
                resource_id="stats_101",
                title="Statistics for Data Science",
                description="Essential statistics concepts for data analysis and machine learning",
                content_type=ContentType.COURSE,
                difficulty_level=DifficultyLevel.INTERMEDIATE,
                skills_covered=["statistics", "probability", "hypothesis testing"],
                prerequisites=["mathematics"],
                duration_minutes=360,
                learning_styles=[LearningStyle.VISUAL, LearningStyle.READING_WRITING],
                rating=4.4,
                tags=["statistics", "data science", "mathematics"]
            ),
            LearningResource(
                resource_id="leadership_1",
                title="Leadership Development Program",
                description="Develop leadership skills and team management capabilities",
                content_type=ContentType.PROJECT,
                difficulty_level=DifficultyLevel.ADVANCED,
                skills_covered=["leadership", "team management", "decision making"],
                prerequisites=["communication", "management"],
                duration_minutes=720,
                learning_styles=[LearningStyle.KINESTHETIC, LearningStyle.AUDITORY],
                rating=4.8,
                tags=["leadership", "advanced", "management"]
            ),
            LearningResource(
                resource_id="ux_design_1",
                title="UX Design Principles",
                description="Learn user experience design fundamentals and best practices",
                content_type=ContentType.COURSE,
                difficulty_level=DifficultyLevel.INTERMEDIATE,
                skills_covered=["ux design", "user research", "prototyping"],
                prerequisites=["design thinking"],
                duration_minutes=240,
                learning_styles=[LearningStyle.VISUAL, LearningStyle.KINESTHETIC],
                rating=4.5,
                tags=["ux design", "design", "user experience"]
            )
        ]
        
        self.resources_database.extend(sample_resources)
    
    def _build_skill_graph(self):
        """Build skill dependency graph"""
        if not nx:
            return
            
        self.skill_graph = nx.DiGraph()
        
        # Add nodes for all skills
        all_skills = set()
        for resource in self.resources_database:
            all_skills.update(resource.skills_covered)
            all_skills.update(resource.prerequisites)
        
        self.skill_graph.add_nodes_from(all_skills)
        
        # Add edges based on prerequisites
        for resource in self.resources_database:
            for skill in resource.skills_covered:
                for prereq in resource.prerequisites:
                    if prereq in all_skills:
                        self.skill_graph.add_edge(prereq, skill)
    
    def assess_learning_style(self, responses: Dict[str, Any]) -> List[LearningStyle]:
        """Assess learning style based on questionnaire responses"""
        style_scores = {
            LearningStyle.VISUAL: 0,
            LearningStyle.AUDITORY: 0,
            LearningStyle.KINESTHETIC: 0,
            LearningStyle.READING_WRITING: 0
        }
        
        # Simple scoring based on preferences
        if responses.get("prefers_videos", False):
            style_scores[LearningStyle.VISUAL] += 3
        if responses.get("prefers_diagrams", False):
            style_scores[LearningStyle.VISUAL] += 2
        if responses.get("prefers_podcasts", False):
            style_scores[LearningStyle.AUDITORY] += 3
        if responses.get("prefers_discussions", False):
            style_scores[LearningStyle.AUDITORY] += 2
        if responses.get("prefers_hands_on", False):
            style_scores[LearningStyle.KINESTHETIC] += 3
        if responses.get("prefers_projects", False):
            style_scores[LearningStyle.KINESTHETIC] += 2
        if responses.get("prefers_reading", False):
            style_scores[LearningStyle.READING_WRITING] += 3
        if responses.get("prefers_notes", False):
            style_scores[LearningStyle.READING_WRITING] += 2
        
        # Return top 2 learning styles
        sorted_styles = sorted(style_scores.items(), key=lambda x: x[1], reverse=True)
        return [style for style, score in sorted_styles[:2] if score > 0]
    
    def analyze_knowledge_gaps(self, learner: LearnerProfile, goal: LearningGoal) -> List[str]:
        """Analyze knowledge gaps for a specific learning goal"""
        gaps = []
        
        # Check if learner has required skills for the goal
        for target_skill in goal.target_skills:
            current_level = learner.current_skills.get(target_skill, 0.0)
            
            # Determine required level based on goal difficulty
            required_levels = {
                DifficultyLevel.BEGINNER: 0.3,
                DifficultyLevel.INTERMEDIATE: 0.6,
                DifficultyLevel.ADVANCED: 0.8,
                DifficultyLevel.EXPERT: 0.9
            }
            
            required_level = required_levels.get(goal.difficulty_level, 0.5)
            
            if current_level < required_level:
                gaps.append(target_skill)
        
        # Check prerequisite skills using skill graph
        if self.skill_graph:
            for skill in goal.target_skills:
                if skill in self.skill_graph:
                    predecessors = list(self.skill_graph.predecessors(skill))
                    for prereq in predecessors:
                        if learner.current_skills.get(prereq, 0.0) < 0.5:
                            gaps.append(prereq)
        
        return list(set(gaps))  # Remove duplicates
    
    def generate_learning_path(self, learner: LearnerProfile, goal: LearningGoal) -> LearningPath:
        """Generate a personalized learning path"""
        path_id = f"path_{learner.learner_id}_{goal.goal_id}_{int(datetime.now().timestamp())}"
        
        # Analyze knowledge gaps
        gaps = self.analyze_knowledge_gaps(learner, goal)
        
        # Find relevant resources
        relevant_resources = self._find_relevant_resources(learner, goal, gaps)
        
        # Score and rank resources
        scored_resources = self._score_resources(learner, relevant_resources)
        
        # Create learning path steps
        steps = self._create_learning_steps(learner, scored_resources)
        
        # Calculate timing
        total_hours = sum(step.resource.duration_minutes for step in steps) / 60
        completion_date = self._calculate_completion_date(learner, total_hours)
        
        # Calculate confidence score
        confidence = self._calculate_confidence_score(learner, goal, steps)
        
        return LearningPath(
            path_id=path_id,
            learner_id=learner.learner_id,
            goal=goal,
            steps=steps,
            estimated_total_hours=int(total_hours),
            estimated_completion_date=completion_date,
            confidence_score=confidence
        )
    
    def _find_relevant_resources(self, learner: LearnerProfile, goal: LearningGoal, gaps: List[str]) -> List[LearningResource]:
        """Find resources relevant to the learning goal and gaps"""
        relevant_resources = []
        
        # Skills to look for (gaps + target skills)
        target_skills = set(gaps + goal.target_skills)
        
        for resource in self.resources_database:
            # Check if resource covers needed skills
            if any(skill in resource.skills_covered for skill in target_skills):
                # Check prerequisites
                prereqs_met = all(
                    learner.current_skills.get(prereq, 0.0) >= 0.3 
                    for prereq in resource.prerequisites
                )
                
                if prereqs_met:
                    relevant_resources.append(resource)
        
        return relevant_resources
    
    def _score_resources(self, learner: LearnerProfile, resources: List[LearningResource]) -> List[Tuple[LearningResource, float]]:
        """Score resources based on learner preferences and relevance"""
        scored_resources = []
        
        for resource in resources:
            score = 0.0
            
            # Learning style match
            style_match = len(set(learner.learning_styles) & set(resource.learning_styles))
            score += style_match * 0.3
            
            # Content type preference
            if resource.content_type in learner.preferred_content_types:
                score += 0.2
            
            # Duration match
            if resource.duration_minutes <= learner.preferred_session_length:
                score += 0.1
            
            # Rating
            if resource.rating:
                score += (resource.rating - 3.0) * 0.1  # Normalize around 3.0
            
            # Interest alignment
            interest_match = len(set(learner.interests) & set(resource.tags))
            score += interest_match * 0.1
            
            # Novelty bonus (not completed before)
            if resource.resource_id not in learner.completed_resources:
                score += 0.1
            
            scored_resources.append((resource, score))
        
        # Sort by score
        scored_resources.sort(key=lambda x: x[1], reverse=True)
        
        return scored_resources
    
    def _create_learning_steps(self, learner: LearnerProfile, scored_resources: List[Tuple[LearningResource, float]]) -> List[LearningPathStep]:
        """Create learning path steps from scored resources"""
        steps = []
        current_time = datetime.now()
        
        for i, (resource, score) in enumerate(scored_resources[:10]):  # Limit to top 10
            # Calculate when this step should be completed
            completion_time = current_time + timedelta(minutes=resource.duration_minutes)
            current_time = completion_time
            
            # Check if prerequisites are met
            prereqs_met = all(
                learner.current_skills.get(prereq, 0.0) >= 0.3 
                for prereq in resource.prerequisites
            )
            
            step = LearningPathStep(
                step_id=f"step_{i+1}_{resource.resource_id}",
                resource=resource,
                estimated_completion_time=completion_time,
                prerequisites_met=prereqs_met,
                importance_score=score,
                personalization_score=score,
                notes=f"Recommended based on {resource.content_type.value} preference and skill gaps"
            )
            
            steps.append(step)
        
        return steps
    
    def _calculate_completion_date(self, learner: LearnerProfile, total_hours: float) -> datetime:
        """Calculate estimated completion date based on learner's availability"""
        weekly_hours = learner.available_hours_per_week
        
        # Adjust for learning pace
        pace_multipliers = {
            "slow": 0.7,
            "medium": 1.0,
            "fast": 1.3
        }
        
        effective_hours = weekly_hours * pace_multipliers.get(learner.learning_pace, 1.0)
        weeks_needed = total_hours / effective_hours if effective_hours > 0 else 1
        
        return datetime.now() + timedelta(weeks=weeks_needed)
    
    def _calculate_confidence_score(self, learner: LearnerProfile, goal: LearningGoal, steps: List[LearningPathStep]) -> float:
        """Calculate confidence score for the learning path"""
        factors = []
        
        # Resource quality
        avg_rating = np.mean([step.resource.rating for step in steps if step.resource.rating])
        if avg_rating:
            factors.append(avg_rating / 5.0)  # Normalize to 0-1
        
        # Learning style match
        style_matches = sum(
            len(set(learner.learning_styles) & set(step.resource.learning_styles))
            for step in steps
        )
        total_possible_matches = len(steps) * len(learner.learning_styles)
        if total_possible_matches > 0:
            factors.append(style_matches / total_possible_matches)
        
        # Prerequisites coverage
        prereq_coverage = sum(1 for step in steps if step.prerequisites_met) / len(steps)
        factors.append(prereq_coverage)
        
        # Path completeness (covers all target skills)
        covered_skills = set()
        for step in steps:
            covered_skills.update(step.resource.skills_covered)
        
        skill_coverage = len(set(goal.target_skills) & covered_skills) / len(goal.target_skills)
        factors.append(skill_coverage)
        
        return np.mean(factors) if factors else 0.5
    
    def add_resource(self, resource: LearningResource):
        """Add a new learning resource to the database"""
        self.resources_database.append(resource)
        
        # Update skill graph
        if self.skill_graph:
            for skill in resource.skills_covered:
                self.skill_graph.add_node(skill)
                for prereq in resource.prerequisites:
                    if prereq not in self.skill_graph:
                        self.skill_graph.add_node(prereq)
                    self.skill_graph.add_edge(prereq, skill)
    
    def get_skill_recommendations(self, current_skills: Dict[str, float], goals: List[LearningGoal]) -> List[str]:
        """Get skill recommendations based on current skills and goals"""
        if not self.skill_graph:
            return []
        
        # Get all target skills from goals
        target_skills = set()
        for goal in goals:
            target_skills.update(goal.target_skills)
        
        # Find skills that are prerequisites for targets but not yet mastered
        recommended_skills = []
        
        for target_skill in target_skills:
            if target_skill in self.skill_graph:
                predecessors = list(self.skill_graph.predecessors(target_skill))
                for prereq in predecessors:
                    if current_skills.get(prereq, 0.0) < 0.5:
                        recommended_skills.append(prereq)
        
        return list(set(recommended_skills))

In [ ]:
class ProgressTracker:
    """Track and analyze learning progress"""
    
    def __init__(self):
        self.progress_data = {}
        self.learning_analytics = {}
    
    def track_progress(self, learner_id: str, step_id: str, completion_percentage: float, 
                      time_spent_minutes: int, performance_score: float = None):
        """Track progress on a learning step"""
        if learner_id not in self.progress_data:
            self.progress_data[learner_id] = {}
        
        self.progress_data[learner_id][step_id] = {
            "completion_percentage": completion_percentage,
            "time_spent_minutes": time_spent_minutes,
            "performance_score": performance_score,
            "last_updated": datetime.now(),
            "attempts": self.progress_data[learner_id].get(step_id, {}).get("attempts", 0) + 1
        }
    
    def get_path_progress(self, learner_id: str, path_id: str, learning_path: LearningPath) -> Dict[str, Any]:
        """Get overall progress for a learning path"""
        if learner_id not in self.progress_data:
            return {"error": "No progress data found"}
        
        learner_progress = self.progress_data[learner_id]
        
        step_progress = []
        total_completion = 0
        total_time_spent = 0
        completed_steps = 0
        
        for step in learning_path.steps:
            step_data = learner_progress.get(step.step_id, {})
            completion = step_data.get("completion_percentage", 0)
            time_spent = step_data.get("time_spent_minutes", 0)
            
            step_progress.append({
                "step_id": step.step_id,
                "resource_title": step.resource.title,
                "completion_percentage": completion,
                "time_spent_minutes": time_spent,
                "performance_score": step_data.get("performance_score"),
                "status": "completed" if completion >= 100 else "in_progress" if completion > 0 else "not_started"
            })
            
            total_completion += completion
            total_time_spent += time_spent
            
            if completion >= 100:
                completed_steps += 1
        
        overall_completion = total_completion / len(learning_path.steps) if learning_path.steps else 0
        
        return {
            "path_id": path_id,
            "learner_id": learner_id,
            "overall_completion_percentage": overall_completion,
            "completed_steps": completed_steps,
            "total_steps": len(learning_path.steps),
            "total_time_spent_minutes": total_time_spent,
            "estimated_time_remaining": max(0, learning_path.estimated_total_hours * 60 - total_time_spent),
            "step_progress": step_progress,
            "updated_at": datetime.now()
        }
    
    def analyze_learning_patterns(self, learner_id: str) -> Dict[str, Any]:
        """Analyze learning patterns for a learner"""
        if learner_id not in self.progress_data:
            return {"error": "No progress data found"}
        
        learner_progress = self.progress_data[learner_id]
        
        # Calculate average completion time
        completion_times = []
        performance_scores = []
        
        for step_data in learner_progress.values():
            if step_data.get("completion_percentage", 0) >= 100:
                completion_times.append(step_data.get("time_spent_minutes", 0))
            
            if step_data.get("performance_score"):
                performance_scores.append(step_data["performance_score"])
        
        # Learning velocity (steps completed per week)
        completed_steps_count = len(completion_times)
        
        # Find peak learning times (if we had timestamp data)
        insights = []
        
        if completion_times:
            avg_completion_time = np.mean(completion_times)
            insights.append(f"Average completion time: {avg_completion_time:.1f} minutes")
            
            if avg_completion_time > 60:
                insights.append("Consider shorter learning sessions")
            elif avg_completion_time < 20:
                insights.append("You might benefit from more comprehensive resources")
        
        if performance_scores:
            avg_performance = np.mean(performance_scores)
            insights.append(f"Average performance score: {avg_performance:.1f}")
            
            if avg_performance > 0.8:
                insights.append("Excellent performance! Consider more challenging content")
            elif avg_performance < 0.6:
                insights.append("Consider reviewing prerequisites or adjusting difficulty")
        
        return {
            "learner_id": learner_id,
            "completed_steps": completed_steps_count,
            "average_completion_time_minutes": np.mean(completion_times) if completion_times else 0,
            "average_performance_score": np.mean(performance_scores) if performance_scores else None,
            "learning_velocity": completed_steps_count,  # Simplified
            "insights": insights,
            "analyzed_at": datetime.now()
        }
    
    def get_skill_proficiency_estimate(self, learner_id: str, skill: str) -> float:
        """Estimate current skill proficiency based on completed learning"""
        if learner_id not in self.progress_data:
            return 0.0
        
        learner_progress = self.progress_data[learner_id]
        
        # This is a simplified estimation - in a real system, you'd use more sophisticated methods
        skill_exposures = 0
        total_performance = 0
        
        for step_data in learner_progress.values():
            if step_data.get("completion_percentage", 0) >= 100:
                # In a real system, you'd check if the step covered this skill
                skill_exposures += 1
                performance = step_data.get("performance_score", 0.5)
                total_performance += performance
        
        if skill_exposures == 0:
            return 0.0
        
        # Estimate proficiency based on performance and exposures
        avg_performance = total_performance / skill_exposures
        exposure_factor = min(1.0, skill_exposures / 5.0)  # Cap at 5 exposures
        
        return avg_performance * exposure_factor
    
    def recommend_path_adjustments(self, learner_id: str, learning_path: LearningPath) -> List[str]:
        """Recommend adjustments to the learning path based on progress"""
        progress = self.get_path_progress(learner_id, learning_path.path_id, learning_path)
        
        if "error" in progress:
            return []
        
        recommendations = []
        
        # Check if learner is falling behind
        if progress["overall_completion_percentage"] < 30:
            recommendations.append("Consider setting smaller, more achievable milestones")
        
        # Check for stuck steps
        stuck_steps = [step for step in progress["step_progress"] 
                      if step["completion_percentage"] > 0 and step["completion_percentage"] < 100]
        
        if len(stuck_steps) > 2:
            recommendations.append("You have several incomplete steps. Consider focusing on one at a time")
        
        # Check time estimates vs actual
        estimated_time = learning_path.estimated_total_hours * 60
        actual_time = progress["total_time_spent_minutes"]
        
        if actual_time > estimated_time * 1.5:
            recommendations.append("Learning is taking longer than expected. Consider adjusting your schedule")
        elif actual_time < estimated_time * 0.5 and progress["overall_completion_percentage"] > 50:
            recommendations.append("You're progressing quickly! Consider adding more advanced content")
        
        return recommendations

In [ ]:
class AdaptiveLearningSystem:
    """Main adaptive learning system that orchestrates all components"""
    
    def __init__(self):
        self.path_generator = LearningPathGenerator()
        self.progress_tracker = ProgressTracker()
        self.learner_profiles = {}
        self.active_paths = {}
    
    def create_learner_profile(self, learner_id: str, name: str, 
                             assessment_responses: Dict[str, Any]) -> LearnerProfile:
        """Create a comprehensive learner profile"""
        
        # Assess learning styles
        learning_styles = self.path_generator.assess_learning_style(assessment_responses)
        
        # Extract preferences
        preferred_content_types = []
        if assessment_responses.get("prefers_videos"):
            preferred_content_types.append(ContentType.VIDEO)
        if assessment_responses.get("prefers_articles"):
            preferred_content_types.append(ContentType.ARTICLE)
        if assessment_responses.get("prefers_interactive"):
            preferred_content_types.append(ContentType.INTERACTIVE)
        if assessment_responses.get("prefers_projects"):
            preferred_content_types.append(ContentType.PROJECT)
        
        # Create profile
        profile = LearnerProfile(
            learner_id=learner_id,
            name=name,
            learning_styles=learning_styles,
            current_skills=assessment_responses.get("current_skills", {}),
            goals=[],
            preferred_content_types=preferred_content_types,
            available_hours_per_week=assessment_responses.get("available_hours_per_week", 5),
            preferred_session_length=assessment_responses.get("preferred_session_length", 60),
            learning_pace=assessment_responses.get("learning_pace", "medium"),
            interests=assessment_responses.get("interests", [])
        )
        
        self.learner_profiles[learner_id] = profile
        return profile
    
    def add_learning_goal(self, learner_id: str, goal: LearningGoal):
        """Add a learning goal to a learner's profile"""
        if learner_id in self.learner_profiles:
            self.learner_profiles[learner_id].goals.append(goal)
    
    def generate_personalized_path(self, learner_id: str, goal_id: str) -> LearningPath:
        """Generate a personalized learning path for a specific goal"""
        if learner_id not in self.learner_profiles:
            raise ValueError(f"Learner profile not found: {learner_id}")
        
        learner = self.learner_profiles[learner_id]
        goal = None
        
        for g in learner.goals:
            if g.goal_id == goal_id:
                goal = g
                break
        
        if not goal:
            raise ValueError(f"Goal not found: {goal_id}")
        
        # Generate the path
        path = self.path_generator.generate_learning_path(learner, goal)
        
        # Store the active path
        self.active_paths[path.path_id] = path
        
        return path
    
    def update_progress(self, learner_id: str, step_id: str, completion_percentage: float,
                       time_spent_minutes: int, performance_score: float = None):
        """Update learning progress and adapt path if needed"""
        # Track progress
        self.progress_tracker.track_progress(
            learner_id, step_id, completion_percentage, time_spent_minutes, performance_score
        )
        
        # Update learner's skill proficiency estimates
        if learner_id in self.learner_profiles:
            learner = self.learner_profiles[learner_id]
            
            # Find the step and update relevant skills
            for path in self.active_paths.values():
                if path.learner_id == learner_id:
                    for step in path.steps:
                        if step.step_id == step_id and completion_percentage >= 100:
                            # Update skill proficiency
                            for skill in step.resource.skills_covered:
                                current_level = learner.current_skills.get(skill, 0.0)
                                improvement = 0.1 * (performance_score or 0.7)
                                learner.current_skills[skill] = min(1.0, current_level + improvement)
                            
                            # Add to completed resources
                            if step.resource.resource_id not in learner.completed_resources:
                                learner.completed_resources.append(step.resource.resource_id)
    
    def get_adaptive_recommendations(self, learner_id: str) -> Dict[str, Any]:
        """Get adaptive recommendations for the learner"""
        if learner_id not in self.learner_profiles:
            return {"error": "Learner profile not found"}
        
        learner = self.learner_profiles[learner_id]
        
        # Get skill recommendations
        skill_recommendations = self.path_generator.get_skill_recommendations(
            learner.current_skills, learner.goals
        )
        
        # Get learning pattern analysis
        learning_patterns = self.progress_tracker.analyze_learning_patterns(learner_id)
        
        # Get path adjustment recommendations
        path_adjustments = []
        for path in self.active_paths.values():
            if path.learner_id == learner_id:
                adjustments = self.progress_tracker.recommend_path_adjustments(learner_id, path)
                path_adjustments.extend(adjustments)
        
        return {
            "learner_id": learner_id,
            "skill_recommendations": skill_recommendations,
            "learning_patterns": learning_patterns,
            "path_adjustments": path_adjustments,
            "generated_at": datetime.now()
        }
    
    def get_dashboard_data(self, learner_id: str) -> Dict[str, Any]:
        """Get comprehensive dashboard data for the learner"""
        if learner_id not in self.learner_profiles:
            return {"error": "Learner profile not found"}
        
        learner = self.learner_profiles[learner_id]
        
        # Get active paths
        active_paths = [path for path in self.active_paths.values() if path.learner_id == learner_id]
        
        # Get progress for each path
        path_progress = []
        for path in active_paths:
            progress = self.progress_tracker.get_path_progress(learner_id, path.path_id, path)
            path_progress.append({
                "path": path,
                "progress": progress
            })
        
        # Calculate overall stats
        total_hours_spent = sum(
            p["progress"].get("total_time_spent_minutes", 0) for p in path_progress
        ) / 60
        
        completed_resources = len(learner.completed_resources)
        total_skills = len(learner.current_skills)
        mastered_skills = sum(1 for level in learner.current_skills.values() if level >= 0.7)
        
        return {
            "learner_id": learner_id,
            "learner_name": learner.name,
            "total_hours_spent": total_hours_spent,
            "completed_resources": completed_resources,
            "total_skills": total_skills,
            "mastered_skills": mastered_skills,
            "active_goals": len(learner.goals),
            "active_paths": len(active_paths),
            "path_progress": path_progress,
            "skill_levels": learner.current_skills,
            "learning_styles": [style.value for style in learner.learning_styles],
            "updated_at": datetime.now()
        }

## Demo: Personalized Learning Path Generator in Action

Let's demonstrate the comprehensive learning path generation system with NLP-powered content analysis and adaptive learning capabilities.

In [ ]:
# Initialize the adaptive learning system
learning_system = AdaptiveLearningSystem()

print("🎓 Personalized Learning Path Generator initialized!")
print(f"📚 Available learning resources: {len(learning_system.path_generator.resources_database)}")

# Display available resources
print("\n📖 Sample Learning Resources:")
for resource in learning_system.path_generator.resources_database[:5]:
    print(f"  • {resource.title} ({resource.content_type.value})")
    print(f"    Skills: {', '.join(resource.skills_covered)}")
    print(f"    Level: {resource.difficulty_level.value}")
    print(f"    Duration: {resource.duration_minutes} minutes")
    print()

In [ ]:
# Create a learner profile through assessment
print("👤 Creating learner profile through assessment...")

# Simulate assessment responses
assessment_responses = {
    "prefers_videos": True,
    "prefers_interactive": True,
    "prefers_hands_on": True,
    "prefers_projects": True,
    "prefers_discussions": False,
    "prefers_reading": False,
    "current_skills": {
        "programming": 0.3,
        "mathematics": 0.6,
        "communication": 0.7,
        "problem solving": 0.5
    },
    "available_hours_per_week": 10,
    "preferred_session_length": 90,
    "learning_pace": "medium",
    "interests": ["technology", "data science", "artificial intelligence", "programming"]
}

# Create learner profile
learner_profile = learning_system.create_learner_profile(
    learner_id="learner_001",
    name="Alex Chen",
    assessment_responses=assessment_responses
)

print(f"\n✅ Profile created for {learner_profile.name}")
print(f"Learning Styles: {[style.value for style in learner_profile.learning_styles]}")
print(f"Preferred Content Types: {[ct.value for ct in learner_profile.preferred_content_types]}")
print(f"Available Hours/Week: {learner_profile.available_hours_per_week}")
print(f"Learning Pace: {learner_profile.learning_pace}")
print(f"Current Skills:")
for skill, level in learner_profile.current_skills.items():
    print(f"  • {skill}: {level:.1%}")
print(f"Interests: {', '.join(learner_profile.interests)}")

In [ ]:
# Create learning goals
print("\n🎯 Creating learning goals...")

# Goal 1: Data Science Mastery
ds_goal = LearningGoal(
    goal_id="goal_ds_001",
    title="Data Science Mastery",
    description="Become proficient in data science including Python, statistics, and machine learning",
    target_skills=["python", "data science", "statistics", "machine learning", "data analysis"],
    difficulty_level=DifficultyLevel.INTERMEDIATE,
    estimated_hours=50,
    priority=9,
    deadline=datetime.now() + timedelta(days=90)
)

# Goal 2: Web Development
web_goal = LearningGoal(
    goal_id="goal_web_001",
    title="Full-Stack Web Development",
    description="Learn full-stack web development from frontend to backend",
    target_skills=["web development", "html", "css", "javascript", "programming"],
    difficulty_level=DifficultyLevel.INTERMEDIATE,
    estimated_hours=40,
    priority=7,
    deadline=datetime.now() + timedelta(days=120)
)

# Add goals to learner profile
learning_system.add_learning_goal("learner_001", ds_goal)
learning_system.add_learning_goal("learner_001", web_goal)

print(f"✅ Added {len(learner_profile.goals)} learning goals:")
for goal in learner_profile.goals:
    print(f"  • {goal.title} ({goal.difficulty_level.value})")
    print(f"    Target Skills: {', '.join(goal.target_skills)}")
    print(f"    Priority: {goal.priority}/10")
    print(f"    Deadline: {goal.deadline.strftime('%Y-%m-%d')}")
    print()

In [ ]:
# Generate personalized learning path for Data Science goal
print("🛤️ Generating personalized learning path for Data Science goal...")

ds_path = learning_system.generate_personalized_path("learner_001", "goal_ds_001")

print(f"\n📊 Generated Learning Path: {ds_path.path_id}")
print(f"Goal: {ds_path.goal.title}")
print(f"Estimated Total Hours: {ds_path.estimated_total_hours}")
print(f"Estimated Completion: {ds_path.estimated_completion_date.strftime('%Y-%m-%d')}")
print(f"Confidence Score: {ds_path.confidence_score:.1%}")
print(f"\n📋 Learning Path Steps ({len(ds_path.steps)} total):")

for i, step in enumerate(ds_path.steps, 1):
    print(f"\n  Step {i}: {step.resource.title}")
    print(f"    Type: {step.resource.content_type.value}")
    print(f"    Level: {step.resource.difficulty_level.value}")
    print(f"    Duration: {step.resource.duration_minutes} minutes")
    print(f"    Skills Covered: {', '.join(step.resource.skills_covered)}")
    print(f"    Prerequisites Met: {'✅' if step.prerequisites_met else '❌'}")
    print(f"    Importance Score: {step.importance_score:.2f}")
    print(f"    Notes: {step.notes}")
    
    if step.resource.prerequisites:
        print(f"    Prerequisites: {', '.join(step.resource.prerequisites)}")

In [ ]:
# Analyze knowledge gaps using NLP
print("\n🔍 Analyzing knowledge gaps using NLP...")

# Analyze gaps for both goals
for goal in learner_profile.goals:
    gaps = learning_system.path_generator.analyze_knowledge_gaps(learner_profile, goal)
    print(f"\n📈 Knowledge Gaps for '{goal.title}':")
    if gaps:
        for gap in gaps:
            current_level = learner_profile.current_skills.get(gap, 0.0)
            print(f"  • {gap}: Current level {current_level:.1%}")
    else:
        print("  ✅ No significant knowledge gaps identified")

# Test NLP content analysis
print("\n🤖 Testing NLP content analysis...")

sample_content = """
This advanced course covers machine learning algorithms including supervised learning, 
unsupervised learning, and deep neural networks. Students will learn to implement 
complex algorithms using Python and TensorFlow, analyze large datasets, and build 
predictive models for real-world applications.
"""

if learning_system.path_generator.nlp_analyzer.nlp:
    # Extract skills from content
    extracted_skills = learning_system.path_generator.nlp_analyzer.extract_skills_from_text(sample_content)
    print(f"\n🔤 Skills extracted from sample content:")
    for skill in extracted_skills:
        print(f"  • {skill}")
    
    # Analyze content difficulty
    difficulty = learning_system.path_generator.nlp_analyzer.analyze_content_difficulty(sample_content)
    print(f"\n📊 Content difficulty analysis: {difficulty.value}")
    
    # Extract learning objectives
    objectives = learning_system.path_generator.nlp_analyzer.extract_learning_objectives(sample_content)
    print(f"\n🎯 Learning objectives extracted:")
    for obj in objectives:
        print(f"  • {obj}")
else:
    print("⚠️ NLP analysis not available (spaCy model not loaded)")

In [ ]:
# Simulate learning progress
print("\n📈 Simulating learning progress...")

# Simulate completion of first few steps
for i, step in enumerate(ds_path.steps[:3]):
    # Simulate different completion scenarios
    if i == 0:
        # First step: completed successfully
        completion = 100
        time_spent = step.resource.duration_minutes + 15  # Took a bit longer
        performance = 0.85
    elif i == 1:
        # Second step: partially completed
        completion = 75
        time_spent = step.resource.duration_minutes // 2
        performance = 0.7
    else:
        # Third step: just started
        completion = 25
        time_spent = 30
        performance = 0.6
    
    learning_system.update_progress(
        learner_id="learner_001",
        step_id=step.step_id,
        completion_percentage=completion,
        time_spent_minutes=time_spent,
        performance_score=performance
    )
    
    print(f"📚 Step {i+1}: {step.resource.title}")
    print(f"  Completion: {completion}%")
    print(f"  Time Spent: {time_spent} minutes")
    print(f"  Performance: {performance:.1%}")
    print()

print("✅ Progress tracking updated!")

In [ ]:
# Get path progress analysis
print("\n📊 Analyzing learning path progress...")

path_progress = learning_system.progress_tracker.get_path_progress(
    "learner_001", ds_path.path_id, ds_path
)

print(f"\n📈 Path Progress Report:")
print(f"Overall Completion: {path_progress['overall_completion_percentage']:.1f}%")
print(f"Completed Steps: {path_progress['completed_steps']}/{path_progress['total_steps']}")
print(f"Total Time Spent: {path_progress['total_time_spent_minutes']} minutes")
print(f"Estimated Time Remaining: {path_progress['estimated_time_remaining']} minutes")

print(f"\n📋 Individual Step Progress:")
for step in path_progress['step_progress']:
    status_emoji = {"completed": "✅", "in_progress": "🔄", "not_started": "⏳"}
    print(f"  {status_emoji[step['status']]} {step['resource_title']}")
    print(f"    Completion: {step['completion_percentage']:.1f}%")
    print(f"    Time Spent: {step['time_spent_minutes']} minutes")
    if step['performance_score']:
        print(f"    Performance: {step['performance_score']:.1%}")
    print()

In [ ]:
# Get adaptive recommendations
print("\n💡 Getting adaptive recommendations...")

recommendations = learning_system.get_adaptive_recommendations("learner_001")

print(f"\n🎯 Skill Recommendations:")
if recommendations['skill_recommendations']:
    for skill in recommendations['skill_recommendations']:
        print(f"  • Focus on developing: {skill}")
else:
    print("  ✅ Current skills align well with your goals")

print(f"\n📊 Learning Pattern Analysis:")
patterns = recommendations['learning_patterns']
if 'error' not in patterns:
    print(f"  Completed Steps: {patterns['completed_steps']}")
    print(f"  Average Completion Time: {patterns['average_completion_time_minutes']:.1f} minutes")
    if patterns['average_performance_score']:
        print(f"  Average Performance: {patterns['average_performance_score']:.1%}")
    print(f"  Learning Velocity: {patterns['learning_velocity']} steps")
    
    print(f"\n💡 Insights:")
    for insight in patterns['insights']:
        print(f"  • {insight}")

print(f"\n🔧 Path Adjustments:")
if recommendations['path_adjustments']:
    for adjustment in recommendations['path_adjustments']:
        print(f"  • {adjustment}")
else:
    print("  ✅ Your learning path is on track!")

In [ ]:
# Get comprehensive dashboard data
print("\n📊 Comprehensive Learning Dashboard...")

dashboard = learning_system.get_dashboard_data("learner_001")

print(f"\n👤 Learner Dashboard for {dashboard['learner_name']}:")
print(f"\n📈 Overall Statistics:")
print(f"  Total Hours Spent: {dashboard['total_hours_spent']:.1f} hours")
print(f"  Completed Resources: {dashboard['completed_resources']}")
print(f"  Total Skills Tracked: {dashboard['total_skills']}")
print(f"  Mastered Skills: {dashboard['mastered_skills']} ({dashboard['mastered_skills']/dashboard['total_skills']*100:.1f}%)")
print(f"  Active Goals: {dashboard['active_goals']}")
print(f"  Active Learning Paths: {dashboard['active_paths']}")

print(f"\n🎯 Current Skill Levels:")
for skill, level in sorted(dashboard['skill_levels'].items(), key=lambda x: x[1], reverse=True):
    bar_length = int(level * 20)
    bar = "█" * bar_length + "░" * (20 - bar_length)
    print(f"  {skill:20} {bar} {level:.1%}")

print(f"\n🧠 Learning Profile:")
print(f"  Learning Styles: {', '.join(dashboard['learning_styles'])}")
print(f"  Available Hours/Week: {learner_profile.available_hours_per_week}")
print(f"  Preferred Session Length: {learner_profile.preferred_session_length} minutes")
print(f"  Learning Pace: {learner_profile.learning_pace}")

print(f"\n📚 Active Learning Paths:")
for path_data in dashboard['path_progress']:
    path = path_data['path']
    progress = path_data['progress']
    print(f"  • {path.goal.title}")
    print(f"    Progress: {progress['overall_completion_percentage']:.1f}%")
    print(f"    Steps: {progress['completed_steps']}/{progress['total_steps']}")
    print(f"    Time Spent: {progress['total_time_spent_minutes']} minutes")
    print()

In [ ]:
# Test adding a custom learning resource
print("\n➕ Adding custom learning resource...")

custom_resource = LearningResource(
    resource_id="custom_deep_learning_1",
    title="Deep Learning Specialization",
    description="Advanced deep learning course covering neural networks, CNNs, RNNs, and transformers",
    content_type=ContentType.COURSE,
    difficulty_level=DifficultyLevel.ADVANCED,
    skills_covered=["deep learning", "neural networks", "tensorflow", "pytorch"],
    prerequisites=["machine learning", "python", "statistics"],
    duration_minutes=900,
    learning_styles=[LearningStyle.VISUAL, LearningStyle.KINESTHETIC],
    rating=4.8,
    tags=["deep learning", "advanced", "neural networks", "ai"]
)

learning_system.path_generator.add_resource(custom_resource)

print(f"✅ Added custom resource: {custom_resource.title}")
print(f"  Skills Covered: {', '.join(custom_resource.skills_covered)}")
print(f"  Prerequisites: {', '.join(custom_resource.prerequisites)}")
print(f"  Duration: {custom_resource.duration_minutes} minutes")
print(f"  Rating: {custom_resource.rating}/5.0")

print(f"\n📚 Total resources now available: {len(learning_system.path_generator.resources_database)}")

In [ ]:
# Generate a new learning path for Web Development goal
print("\n🌐 Generating learning path for Web Development goal...")

web_path = learning_system.generate_personalized_path("learner_001", "goal_web_001")

print(f"\n💻 Web Development Learning Path: {web_path.path_id}")
print(f"Goal: {web_path.goal.title}")
print(f"Estimated Total Hours: {web_path.estimated_total_hours}")
print(f"Estimated Completion: {web_path.estimated_completion_date.strftime('%Y-%m-%d')}")
print(f"Confidence Score: {web_path.confidence_score:.1%}")

print(f"\n📋 Top Learning Steps:")
for i, step in enumerate(web_path.steps[:3], 1):
    print(f"  {i}. {step.resource.title}")
    print(f"     Type: {step.resource.content_type.value}")
    print(f"     Duration: {step.resource.duration_minutes} minutes")
    print(f"     Skills: {', '.join(step.resource.skills_covered)}")
    print()

print(f"\n🎉 Personalized Learning Path Generator Demo Complete!")
print("\nSuccessfully demonstrated:")
print("  ✅ NLP-powered content analysis and skill extraction")
print("  ✅ Learning style assessment and profiling")
print("  ✅ Knowledge gap analysis")
print("  ✅ Personalized learning path generation")
print("  ✅ Adaptive progress tracking")
print("  ✅ Intelligent recommendations")
print("  ✅ Comprehensive learning analytics")
print("  ✅ Skill dependency mapping")
print("  ✅ Multi-modal content recommendation")

## Production Deployment Considerations

### spaCy NLP Configuration
```python
# Production spaCy setup
spacy_config = {
    "models": {
        "base": "en_core_web_sm",
        "advanced": "en_core_web_lg",
        "domain_specific": "custom_education_model"
    },
    "custom_components": [
        "skill_extractor",
        "difficulty_analyzer",
        "objective_parser"
    ],
    "performance_settings": {
        "batch_size": 32,
        "max_length": 1000000,
        "disable_components": ["parser", "tagger"]
    }
}
```

### Scalability Features
- **Intelligent Content Analysis**: NLP-powered skill extraction and difficulty assessment
- **Adaptive Learning Engine**: Real-time path adjustments based on performance
- **Multi-Modal Learning**: Support for various content types and learning styles
- **Knowledge Graph**: Skill dependency mapping and prerequisite analysis
- **Progress Analytics**: Comprehensive learning pattern analysis
- **Recommendation Engine**: Personalized content and skill suggestions

### Infrastructure Requirements
- **Database**: PostgreSQL with vector extensions for similarity search
- **ML Pipeline**: scikit-learn for clustering and recommendation algorithms
- **NLP Processing**: spaCy with custom education domain models
- **Caching**: Redis for frequently accessed learning paths
- **API Gateway**: RESTful API for learning management systems
- **Analytics**: Real-time dashboards for learner progress tracking

This Personalized Learning Path Generator showcases the power of combining NLP with adaptive learning algorithms to create truly individualized educational experiences that evolve with each learner's progress and preferences.